In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2-1.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    torch_dtype = torch.float16
)
model = model.to("cuda:0")

In [ ]:
print(next(model.parameters()).device)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

df = pd.read_csv("/kaggle/input/review-txt/review_txt", sep='\t', header=None, names = ['label', '내용'])
df = df[df['label'].isin(['부정', '긍정', '중립'])]

train_df, valid_df = train_test_split(
    df,
    test_size = 0.3,
    stratify = df['label'], # label 기준으로 계층 나누기
    random_state = 42
)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

dataset = DatasetDict({
    'train': train_dataset,
    'validation': valid_dataset
})

def tokenize_fn(example):
    full_prompt = [f"질문 : {contents}\n정답 : {label}" for contents, label in zip(example['내용'], example['label'])]    
    return tokenizer(full_prompt, truncation=True, padding='max_length', max_length=256)

tokenized_datasets = dataset.map(tokenize_fn, batched=True)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    eval_strategy="steps",
    logging_steps=10,
    save_steps=500,
    disable_tqdm=False,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

trainer.train()